# Notebook 04: Two-Zone Model with UltraNest

**Object:** MG3 J163554+3629 (z = 3.648)
**Depends on:** `../../data/SED.txt` — builds everything from scratch.
**Requires:** Notebook 03 to have run first (uses its Minuit best-fit values as starting point for priors).

**Structure:**
- Part 1: Rebuild data + two-zone model from scratch (avoids pkl R dependency bug)
- Part 2: Setup free parameters + likelihood
- Part 3: Run UltraNest
- Part 4: Results — corner plot, best-fit SED, posterior envelope

**Outputs saved to `../../results/`:**
- `ultranest_output_2zone/` — UltraNest chain files
- `models/ultranest_samples_2zone.npy`
- `models/ultranest_results_2zone.npy`
- `plots/ultranest_corner_2zone.png`
- `plots/ultranest_bestfit_sed_2zone.png`
- `plots/ultranest_envelope_2zone.png`

---
## Part 1: Rebuild data + two-zone model from scratch

In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
import corner
import ultranest
import ultranest.stepsampler
import os
%matplotlib inline

from jetset.data_loader import Data, ObsData
from jetset.sed_shaper import SEDShape
from jetset.jet_model import Jet
from jetset.obs_constrain import ObsConstrain
from jetset.model_manager import FitModel
from jetset.template_2Dmodel import EBLAbsorptionTemplate

os.makedirs('../../results/models', exist_ok=True)
os.makedirs('../../results/plots',  exist_ok=True)
print('Imports done.')

Imports done.


In [2]:
# ============================================================
# LOAD DATA FROM SCRATCH
# Use unbinned data (same as notebook 02 UltraNest)
# ============================================================
d = np.genfromtxt('../../data/SED.txt')
d = d[d[:, 0] != 1.709E22]

data_obj = Data(n_rows=d.shape[0])
data_obj.set_field('x', d[:, 0])
data_obj.set_field('y', d[:, 1])
data_obj.set_field('dy', d[:, 2])
data_obj.set_meta_data('z',          3.647763)
data_obj.set_meta_data('UL_CL',      0.95)
data_obj.set_meta_data('restframe',  'obs')
data_obj.set_meta_data('data_scale', 'lin-lin')
data_obj.set_meta_data('obj_name',   'MG3J163554+3629')

UL_freq = [1.324e25, 7.646e25, 4.188e24, 4.188e22, 2.417e13, 1.318e13]

sed_data = ObsData(data_table=data_obj)
sed_data.filter_data_set('-1', exclude=True)
sed_data.add_systematics(0.15, [10.**6, 10.**29])
for freq in UL_freq:
    mask = np.isclose(sed_data.table['nu_data'], freq, rtol=0.05)
    sed_data.table['UL'][mask] = True

# Extract data arrays for likelihood
nu_obs   = np.array(sed_data.table['nu_data'])
flux_obs = np.array(sed_data.table['nuFnu_data'])
flux_err = np.array(sed_data.table['dnuFnu_data'])
is_ul    = np.array(sed_data.table['UL'], dtype=bool)

# Include radio points (1e8 Hz lower bound)
fit_mask = (nu_obs >= 1e8) & (nu_obs <= 1e29)
nu_obs   = nu_obs[fit_mask]
flux_obs = flux_obs[fit_mask]
flux_err = flux_err[fit_mask]
is_ul    = is_ul[fit_mask]

print(f'Data points in fit: {len(nu_obs)}')
print(f'Radio points (<1e10 Hz): {nu_obs[nu_obs < 1e10]}')

---> excluding  data_set/s ['-1']
filter -1 45
current datasets
dataset 0.0
---> data sets left after filtering None
---> data len after filtering=45
Data points in fit: 45
Radio points (<1e10 Hz): [1.40e+09 1.40e+09 1.40e+09 1.40e+09 4.83e+09 4.85e+09 4.85e+09 5.00e+09]


In [3]:
# ============================================================
# BUILD ZONE 1 FROM SCRATCH
# Use best-fit values from notebook 03 Minuit as starting point
# ============================================================
my_shape = SEDShape(sed_data)
my_shape.eval_indices(silent=True)
my_shape.sync_fit(check_BBB_template=True, check_host_gal_template=False,
                   use_log_par=True, minimizer='lsb', fit_range=[9,16], silent=True)
my_shape.IC_fit(fit_range=[16,26], minimizer='minuit', silent=True)

sed_obspar = ObsConstrain(
    theta=3, bulk_factor=18.5, B_range=[0.02,0.08],
    distr_e='bkn', t_var_sec=7*86400, nu_cut_IR=1E9,
    SEDShape=my_shape
)
zone1_jet = sed_obspar.constrain_SSC_EC_model(
    electron_distribution_log_values=False,
    EC_components_list=['EC_DT','EC_BLR','EC_Disk'],
    R_H=1.5E18, disk_type='MultiBB', silent=True
)

# Set to Minuit best-fit values from notebook 03
# UPDATE THESE with your actual notebook 03 best-fit values
zone1_jet.set_par('L_Disk',   val=1.5817e46)
zone1_jet.set_par('M_BH',     val=1.9077e9)
zone1_jet.set_par('accr_eff', val=0.083)
zone1_jet.set_par('T_DT',     val=600.9)
zone1_jet.set_par('tau_DT',   val=0.1)
zone1_jet.set_par('tau_BLR',  val=0.1)
zone1_jet.parameters.p.val           = 1.3
zone1_jet.parameters.p_1.val         = 2.9
zone1_jet.parameters.gamma_break.val = 400.0
zone1_jet.parameters.N.val           = 1000.0
zone1_jet.parameters.B.val           = 0.05
zone1_jet.parameters.gmin.val        = 2.0
zone1_jet.parameters.gmax.val        = 16000.0
zone1_jet.parameters.theta.val       = 3.0
zone1_jet.parameters.BulkFactor.val  = 15.0
zone1_jet.parameters.R_inner_Sw.val  = 10.1
zone1_jet.parameters.R_ext_Sw.val    = 846.0

# Set R_H FIRST, then add dependency
zone1_jet.parameters.R_H.val = 1.5e18
zone1_jet.add_user_par(name='theta_open', val=3, units='deg')
zone1_jet.make_dependent_par(
    par='R', depends_on=['R_H','theta_open'],
    par_expr='np.tan(np.radians(theta_open))*R_H'
)
zone1_jet.eval()

assert zone1_jet.parameters.R.val > 0, 'Zone 1 R is negative!'
print(f'Zone 1: R_H={zone1_jet.parameters.R_H.val:.3e}, R={zone1_jet.parameters.R.val:.3e}')
print(f'Zone 1: R_DT={zone1_jet.parameters.R_DT.val:.3e}')


*** evaluating spectral indices for data ***


*** Log-Polynomial fitting of the synchrotron component ***
---> first blind fit run,  fit range: [9, 16]
---> class:  ISP

--> class:  LSP




model name,name,val,bestfit val,err +,err -,start val,fit range min,fit range max,frozen
LogParabolaEp,b,-8.256408e-02,-8.256408e-02,2.584721e-02,--,-6.224913e-02,-1.000000e+01,0.000000e+00,False
LogParabolaEp,Ep,1.330904e+01,1.330904e+01,5.717990e-01,--,1.413585e+01,0.000000e+00,3.000000e+01,False
LogParabolaEp,Sp,-1.333490e+01,-1.333490e+01,7.469589e-02,--,-1.320459e+01,-3.000000e+01,0.000000e+00,False
BBB,nuFnu_p_BBB,-1.338371e+01,-1.338371e+01,1.806069e-01,--,-1.320459e+01,-1.520459e+01,-1.120459e+01,False
BBB,nu_scale,4.312914e-02,4.312914e-02,6.422704e-03,--,0.000000e+00,-5.000000e-01,5.000000e-01,False


---> sync       nu_p=+1.330904e+01 (err=+5.717990e-01)  nuFnu_p=-1.333490e+01 (err=+7.469589e-02) curv.=-8.256408e-02 (err=+2.584721e-02)


*** Log-Polynomial fitting of the IC component ***
---> fit range: [16, 26]
---> LogCubic fit
====> simplex
====> migrad
====> simplex
====> migrad
====> simplex
====> migrad




model name,name,val,bestfit val,err +,err -,start val,fit range min,fit range max,frozen
LogCubic,b,-1.362857e-01,-1.362857e-01,2.137025e-02,--,-1.000000e+00,-1.000000e+01,0.000000e+00,False
LogCubic,c,-1.412340e-02,-1.412340e-02,3.894639e-03,--,-1.000000e+00,-1.000000e+01,1.000000e+01,False
LogCubic,Ep,2.206163e+01,2.206163e+01,1.234883e-01,--,2.324243e+01,0.000000e+00,3.000000e+01,False
LogCubic,Sp,-1.186960e+01,-1.186960e+01,1.075056e-01,--,-1.000000e+01,-3.000000e+01,0.000000e+00,False


---> IC         nu_p=+2.206163e+01 (err=+1.234883e-01)  nuFnu_p=-1.186960e+01 (err=+1.075056e-01) curv.=-1.362857e-01 (err=+2.137025e-02)


***  constrains parameters from observable ***

===> setting C threads to 8
adding par: L_Disk to  R_BLR_in
==> par R_BLR_in is depending on ['L_Disk'] according to expr:   R_BLR_in =
3E17*(L_Disk/1E46)**0.5
adding par: R_BLR_in to  R_BLR_out
==> par R_BLR_out is depending on ['R_BLR_in'] according to expr:   R_BLR_out =
R_BLR_in*1.1
adding par: L_Disk to  R_DT
==> par R_DT is depending on ['L_Disk'] according to expr:   R_DT =
2E19*(L_Disk/1E46)**0.5


model name,name,par type,units,val,phys. bound. min,phys. bound. max,log,frozen
jet_leptonic,R,region_size,cm,3.721835e+16,1.000000e+03,1.000000e+30,False,False
jet_leptonic,R_H,region_position,cm,1.500000e+18,0.000000e+00,--,False,True
jet_leptonic,B,magnetic_field,gauss,7.620413e-02,0.000000e+00,--,False,False
jet_leptonic,NH_cold_to_rel_e,cold_p_to_rel_e_ratio,,1.000000e+00,0.000000e+00,--,False,True
jet_leptonic,theta,jet-viewing-angle,deg,3.000000e+00,0.000000e+00,9.000000e+01,False,False
jet_leptonic,BulkFactor,jet-bulk-factor,lorentz-factor*,1.850000e+01,1.000000e+00,1.000000e+05,False,False
jet_leptonic,z_cosm,redshift,,3.647763e+00,0.000000e+00,--,False,False
jet_leptonic,gmin,low-energy-cut-off,lorentz-factor*,2.937958e+01,1.000000e+00,1.000000e+09,False,False
jet_leptonic,gmax,high-energy-cut-off,lorentz-factor*,2.686264e+04,1.000000e+00,1.000000e+15,False,False
jet_leptonic,N,emitters_density,1 / cm3,1.717180e+03,0.000000e+00,--,False,False




adding par: R_H to  R
adding par: theta_open to  R
==> par R is depending on ['R_H', 'theta_open'] according to expr:   R =
np.tan(np.radians(theta_open))*R_H
Zone 1: R_H=1.500e+18, R=7.861e+16
Zone 1: R_DT=2.515e+19


In [4]:
# ============================================================
# BUILD ZONE 2 FROM SCRATCH
# ============================================================
zone2_jet = Jet(
    name='jet_extended',
    electron_distribution='bkn',
    electron_distribution_log_values=False,
    beaming_expr='bulk_theta'
)
zone2_jet.add_EC_component(
    EC_components_list=['EC_DT','EC_BLR','EC_Disk'],
    disk_type='MultiBB'
)

for par_name in ['z_cosm','L_Disk','M_BH','accr_eff','R_inner_Sw','R_ext_Sw',
                  'T_DT','tau_DT','tau_BLR','theta','BulkFactor']:
    zone2_jet.set_par(par_name, val=zone1_jet.parameters.get_par_by_name(par_name).val)

zone2_jet.add_user_par(name='theta_open', val=3, units='deg')
zone2_jet.make_dependent_par(
    par='R', depends_on=['R_H','theta_open'],
    par_expr='np.tan(np.radians(theta_open))*R_H'
)

R_DT_z1 = zone1_jet.parameters.R_DT.val
zone2_jet.set_par('R_H', val=5.0 * R_DT_z1)

zone2_jet.set_par('p',           val=zone1_jet.parameters.p.val)
zone2_jet.set_par('p_1',         val=zone1_jet.parameters.p_1.val)
zone2_jet.set_par('gamma_break', val=zone1_jet.parameters.gamma_break.val)
zone2_jet.set_par('gmin',        val=zone1_jet.parameters.gmin.val)
zone2_jet.set_par('gmax',        val=zone1_jet.parameters.gmax.val)
zone2_jet.set_par('B',           val=zone1_jet.parameters.B.val / 10.0)
zone2_jet.set_par('N',           val=zone1_jet.parameters.N.val * 1e-4)

zone2_jet.spectral_components.EC_DT.state  = 'off'
zone2_jet.spectral_components.EC_BLR.state = 'off'
zone2_jet.spectral_components.EC_Disk.state = 'off'
zone2_jet.spectral_components.DT.state     = 'off'
zone2_jet.spectral_components.Disk.state   = 'off'
zone2_jet.eval()

assert zone2_jet.parameters.R.val > 0, 'Zone 2 R is negative!'
print(f'Zone 2: R_H={zone2_jet.parameters.R_H.val:.3e}, R={zone2_jet.parameters.R.val:.3e}')

===> setting C threads to 8
adding par: R_H to  R
adding par: theta_open to  R
==> par R is depending on ['R_H', 'theta_open'] according to expr:   R =
np.tan(np.radians(theta_open))*R_H
Zone 2: R_H=1.258e+20, R=6.591e+18


In [5]:
# ============================================================
# BUILD COMPOSITE MODEL
# ============================================================
fit_model = FitModel(jet=zone1_jet, name='TwoZone_Model')
fit_model.add_component(zone2_jet)

ebl = EBLAbsorptionTemplate.from_name('Franceschini_2008')
fit_model.add_component(ebl)

linked_pars = ['z_cosm','theta','BulkFactor','theta_open',
               'L_Disk','M_BH','accr_eff',
               'T_DT','tau_DT','tau_BLR','R_inner_Sw','R_ext_Sw']
for par in linked_pars:
    try:
        fit_model.link_par(par_name=par,
                            from_model='jet_extended',
                            to_model='jet_leptonic')
    except Exception as e:
        print(f'  Could not link {par}: {e}')

fit_model.link_par(par_name='z_cosm',
                    from_model='Franceschini_2008',
                    to_model='jet_leptonic')
fit_model.composite_expr = '(jet_leptonic + jet_extended) * Franceschini_2008'
fit_model.eval()
print('Two-zone composite model built successfully.')

adding par: z_cosm to  z_cosm
adding par: theta to  theta
adding par: BulkFactor to  BulkFactor
adding par: theta_open to  theta_open
adding par: L_Disk to  L_Disk
adding par: M_BH to  M_BH
adding par: accr_eff to  accr_eff
adding par: T_DT to  T_DT
adding par: tau_DT to  tau_DT
adding par: tau_BLR to  tau_BLR
adding par: R_inner_Sw to  R_inner_Sw
adding par: R_ext_Sw to  R_ext_Sw
adding par: z_cosm to  z_cosm
Two-zone composite model built successfully.


---
## Part 2: Setup free parameters and likelihood

In [6]:
# ============================================================
# FREEZE ALL, then free chosen subset
# ============================================================
for par in fit_model.jet_leptonic.parameters.par_array:
    par.frozen = True
for par in fit_model.jet_extended.parameters.par_array:
    par.frozen = True

# Fix T_DT at best-fit value
T_DT_FIXED = fit_model.jet_leptonic.parameters.T_DT.val
print(f'T_DT fixed at {T_DT_FIXED:.2f} K')

# Free parameters: Zone 1 core physics + Zone 2 N and B
free_params_zone1 = ['B', 'BulkFactor', 'N', 'gamma_break', 'M_BH', 'L_Disk',
                      'p', 'p_1', 'R_H', 'gmin']
free_params_zone2 = ['N', 'B']

for name in free_params_zone1:
    fit_model.jet_leptonic.parameters.get_par_by_name(name).frozen = False
for name in free_params_zone2:
    fit_model.jet_extended.parameters.get_par_by_name(name).frozen = False

# Use tuples (jet_name, param_name) to handle N and B existing on both jets
free_params = (
    [('jet_leptonic', n) for n in free_params_zone1] +
    [('jet_extended', n) for n in free_params_zone2]
)
param_labels = [f'{jet}.{name}' for jet, name in free_params]

# Prior bounds — from notebook 03 fit_range values
prior_mins = np.array([
    0.010,   # jet_leptonic.B
    10.0,    # jet_leptonic.BulkFactor
    400.0,   # jet_leptonic.N
    200.0,   # jet_leptonic.gamma_break
    5.0e8,   # jet_leptonic.M_BH
    1.0e46,  # jet_leptonic.L_Disk
    1.0,     # jet_leptonic.p
    2.5,     # jet_leptonic.p_1
    5.0e17,  # jet_leptonic.R_H
    1.0,     # jet_leptonic.gmin
    1.0e-3,  # jet_extended.N
    1.0e-4,  # jet_extended.B
])
prior_maxs = np.array([
    0.100,   # jet_leptonic.B
    25.0,    # jet_leptonic.BulkFactor
    2000.0,  # jet_leptonic.N
    3000.0,  # jet_leptonic.gamma_break
    2.0e9,   # jet_leptonic.M_BH
    5.0e46,  # jet_leptonic.L_Disk
    1.8,     # jet_leptonic.p
    3.8,     # jet_leptonic.p_1
    3.0e18,  # jet_leptonic.R_H
    50.0,    # jet_leptonic.gmin
    10.0,    # jet_extended.N
    5.0e-2,  # jet_extended.B
])

print(f'Free parameters ({len(free_params)}):')
for (jet, name), lo, hi in zip(free_params, prior_mins, prior_maxs):
    print(f'  {jet}.{name:15s}: [{lo:.3e}, {hi:.3e}]')

T_DT fixed at 600.90 K
Free parameters (12):
  jet_leptonic.B              : [1.000e-02, 1.000e-01]
  jet_leptonic.BulkFactor     : [1.000e+01, 2.500e+01]
  jet_leptonic.N              : [4.000e+02, 2.000e+03]
  jet_leptonic.gamma_break    : [2.000e+02, 3.000e+03]
  jet_leptonic.M_BH           : [5.000e+08, 2.000e+09]
  jet_leptonic.L_Disk         : [1.000e+46, 5.000e+46]
  jet_leptonic.p              : [1.000e+00, 1.800e+00]
  jet_leptonic.p_1            : [2.500e+00, 3.800e+00]
  jet_leptonic.R_H            : [5.000e+17, 3.000e+18]
  jet_leptonic.gmin           : [1.000e+00, 5.000e+01]
  jet_extended.N              : [1.000e-03, 1.000e+01]
  jet_extended.B              : [1.000e-04, 5.000e-02]


In [7]:
# ============================================================
# LIKELIHOOD AND PRIOR
# ============================================================
def get_composite_flux(nu_target):
    fit_model.eval()
    nu_grid   = fit_model.SED.nu.value
    flux_grid = fit_model.SED.nuFnu.value
    good      = np.isfinite(flux_grid) & np.isfinite(nu_grid) & (nu_grid > 0)
    nu_g, fl_g = nu_grid[good], flux_grid[good]
    idx = np.argsort(nu_g)
    return np.interp(nu_target, nu_g[idx], fl_g[idx], left=0., right=0.)

def prior_transform(cube):
    return prior_mins + cube * (prior_maxs - prior_mins)

def log_likelihood(params):
    try:
        for (jet_name, par_name), val in zip(free_params, params):
            getattr(fit_model, jet_name).set_par(par_name, val=val)
        model_flux = get_composite_flux(nu_obs)
        if np.any(~np.isfinite(model_flux)):
            return -1e100
        chi2 = 0.0
        for i in range(len(nu_obs)):
            if flux_err[i] <= 0:
                continue
            if is_ul[i]:
                if model_flux[i] > flux_obs[i]:
                    chi2 += ((model_flux[i] - flux_obs[i]) / flux_err[i])**2
            else:
                if model_flux[i] > 0:
                    chi2 += ((model_flux[i] - flux_obs[i]) / flux_err[i])**2
        return -0.5 * chi2 if np.isfinite(chi2) else -1e100
    except Exception:
        return -1e100

# Sanity check at starting point
best_vals = np.array([
    getattr(fit_model, jet).parameters.get_par_by_name(name).val
    for jet, name in free_params
])
ll_test = log_likelihood(best_vals)
print(f'logL at starting point: {ll_test:.4f}  (should not be -1e100)')
assert ll_test > -1e99, 'Likelihood broken — check model evaluation.'

logL at starting point: -236.2148  (should not be -1e100)


---
## Part 3: Run UltraNest

In [8]:
# ============================================================
# RUN ULTRANEST
# ============================================================
sampler = ultranest.ReactiveNestedSampler(
    param_names = param_labels,
    loglike     = log_likelihood,
    transform   = prior_transform,
    log_dir     = '../../results/ultranest_output_2zone',
    resume      = 'overwrite'
)

sampler.stepsampler = ultranest.stepsampler.SliceSampler(
    nsteps             = 2 * len(free_params),
    generate_direction = ultranest.stepsampler.generate_mixture_random_direction
)

print(f'Running UltraNest: {len(free_params)} free parameters, 200 live points...')
results = sampler.run(
    min_num_live_points = 200,
    max_iters           = 100000,
    show_status         = True,
    viz_callback        = False
)

sampler.print_results()
print(f'\nlog Z (2-zone) = {results["logz"]:.3f} +/- {results["logzerr"]:.3f}')
print('Compare to log Z (1-zone) from notebook 02 for Bayesian model comparison.')

# Save
samples = results['samples']
np.save('../../results/models/ultranest_samples_2zone.npy', samples)
np.save('../../results/models/ultranest_results_2zone.npy', results, allow_pickle=True)
print('Saved: ultranest_samples_2zone.npy')

Running UltraNest: 12 free parameters, 200 live points...
[ultranest] Sampling 200 live points from prior ...


KeyboardInterrupt: 

---
## Part 4: Results — corner plot, best-fit SED, posterior envelope

In [ ]:
# ============================================================
# POSTERIOR SUMMARY
# ============================================================
samples  = results['samples']
post_med = np.median(samples, axis=0)
post_lo  = np.percentile(samples, 16, axis=0)
post_hi  = np.percentile(samples, 84, axis=0)

print('\n=== Two-zone UltraNest posterior (median +/- 1sigma) ===')
for (jet, name), med, lo, hi in zip(free_params, post_med, post_lo, post_hi):
    print(f'  {jet}.{name:15s}: {med:.4e}  +{hi-med:.4e}  -{med-lo:.4e}')
print(f'\nlog Z = {results["logz"]:.3f} +/- {results["logzerr"]:.3f}')

In [ ]:
# ============================================================
# CORNER PLOT
# ============================================================
fig = corner.corner(
    samples,
    labels=param_labels,
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_fmt='.3g',
    title_kwargs=dict(fontsize=9),
    label_kwargs=dict(fontsize=9),
    smooth=1.0
)
plt.suptitle('MG3 J163554+3629 — UltraNest posterior (2-zone)', y=1.01, fontsize=12)
plt.savefig('../../results/plots/ultranest_corner_2zone.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ultranest_corner_2zone.png')

In [ ]:
# ============================================================
# BEST-FIT SED (posterior median)
# ============================================================
for (jet_name, par_name), val in zip(free_params, post_med):
    getattr(fit_model, jet_name).set_par(par_name, val=val)
fit_model.eval()

p = fit_model.plot_model(sed_data=sed_data)
p.add_residual_plot(fit_model, sed_data)
p.setlim(y_min=1E-16, x_min=1E8)
plt.suptitle('MG3 J163554+3629 — UltraNest posterior median (2-zone)')
plt.savefig('../../results/plots/ultranest_bestfit_sed_2zone.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ultranest_bestfit_sed_2zone.png')

In [ ]:
# ============================================================
# POSTERIOR ENVELOPE (68% credible band)
# ============================================================
n_draws  = 100
rng      = np.random.default_rng(42)
draw_idx = rng.choice(len(samples), size=min(n_draws, len(samples)), replace=False)

nu_plot     = fit_model.SED.nu.value
flux_draws  = []

for idx in draw_idx:
    for (jet_name, par_name), val in zip(free_params, samples[idx]):
        getattr(fit_model, jet_name).set_par(par_name, val=val)
    fit_model.eval()
    flux_draws.append(fit_model.SED.nuFnu.value.copy())

flux_draws = np.array(flux_draws)
flux_lo    = np.percentile(flux_draws, 16, axis=0)
flux_med   = np.percentile(flux_draws, 50, axis=0)
flux_hi    = np.percentile(flux_draws, 84, axis=0)
good       = flux_med > 0

fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(nu_plot[good], flux_lo[good], flux_hi[good],
                alpha=0.35, color='darkorange', label='posterior 68% band')
ax.loglog(nu_plot[good], flux_med[good],
          color='darkorange', lw=2, label='median model')

# Data
det_mask = ~np.array(sed_data.table['UL'], dtype=bool)
ul_mask  =  np.array(sed_data.table['UL'], dtype=bool)
nu_d     = np.array(sed_data.table['nu_data'])
fl_d     = np.array(sed_data.table['nuFnu_data'])
err_d    = np.array(sed_data.table['dnuFnu_data'])

ax.errorbar(nu_d[det_mask], fl_d[det_mask], yerr=err_d[det_mask],
            fmt='o', color='navy', ms=4, lw=0.8, capsize=2, label='data', zorder=5)
ax.errorbar(nu_d[ul_mask], fl_d[ul_mask], yerr=fl_d[ul_mask]*0.2,
            fmt='v', color='gray', ms=4, uplims=True, label='upper limits')

ax.set_xlim(1e8, 1e27); ax.set_ylim(1e-16, 1e-10)
ax.set_xlabel(r'$\nu$ [Hz]', fontsize=13)
ax.set_ylabel(r'$\nu F_\nu$ [erg cm$^{-2}$ s$^{-1}$]', fontsize=13)
ax.set_title('MG3 J163554+3629 — UltraNest posterior envelope (2-zone)', fontsize=12)
ax.legend(fontsize=10); ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('../../results/plots/ultranest_envelope_2zone.png', dpi=150, bbox_inches='tight')
plt.show()

print('Saved: ultranest_envelope_2zone.png')
print(f'\nlog Z (2-zone) = {results["logz"]:.3f} +/- {results["logzerr"]:.3f}')
print('Run notebook 02 to get log Z (1-zone) for Bayesian model comparison.')